In [ ]:
import json
from pathlib import Path

required_names = ["X_train_enc", "X_train_scaled", "y_train", "class_weight_dict"]
missing = [name for name in required_names if name not in globals()]

if missing:
    notebook_path = Path("../preprocessing.ipynb")
    with notebook_path.open(encoding="utf-8") as f:
        preprocessing_nb = json.load(f)

    for cell in preprocessing_nb["cells"]:
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        exec(source, globals())

        if all(name in globals() for name in required_names):
            break

missing = [name for name in required_names if name not in globals()]
if missing:
    raise NameError(f"Missing preprocessing variables: {missing}")

X_train_enc = globals()["X_train_enc"]
X_train_scaled = globals()["X_train_scaled"]
y_train = globals()["y_train"]
class_weight_dict = globals()["class_weight_dict"]


## Machine Learning Model Selection

Three supervised ML models are trained and compared for this binary classification task: **Decision Tree**, **Random Forest**, and **Logistic Regression**.


### Model 1: Decision Tree


In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=10,
    min_samples_split=40,
    min_samples_leaf=20,
    class_weight=class_weight_dict,
    random_state=55
)
dt.fit(X_train_enc, y_train)

print("Decision Tree trained successfully")
print(f"Criterion: {dt.criterion}")
print(f"Tree depth: {dt.get_depth()}")
print(f"Number of leaves: {dt.get_n_leaves()}")
print(f"Features used: {dt.n_features_in_}")

Decision Tree trained successfully
Criterion: entropy
Tree depth: 10
Number of leaves: 314
Features used: 6


### Model 2: Random Forest


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, class_weight=class_weight_dict, random_state=42)
rf.fit(X_train_enc, y_train)

print("Random Forest trained successfully")
print(f"Number of trees: {rf.n_estimators}")
print(f"Criterion: {rf.criterion}")
print(f"Features used: {rf.n_features_in_}")

Random Forest trained successfully
Number of trees: 200
Criterion: gini
Features used: 6


### Model 3: Logistic Regression


In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(solver='lbfgs', max_iter=300, C=1.0, class_weight=class_weight_dict, random_state=42)
lr.fit(X_train_scaled, y_train)

print("Logistic Regression trained successfully")
print(f"Solver: {lr.solver}")
print(f"Max iterations: {lr.max_iter}")
print(f"Iterations used: {lr.n_iter_[0]}")
print(f"Regularization C: {lr.C}")
print(f"Features used: {lr.n_features_in_}")

Logistic Regression trained successfully
Solver: lbfgs
Max iterations: 300
Iterations used: 13
Regularization C: 1.0
Features used: 6


## Selected Models

**Logistic Regression** is selected as the primary classification model because its linear architecture and regularization are a strong fit for a low-dimensional tabular dataset with 6 structured features.

**Random Forest** is additionally selected because its ensemble architecture can capture non-linear interactions and provide feature importance scores for candidate feedback.
